# analise do pipeline medalhao de dados de taxi em NY

este notebook conecta diretamente ao banco PostgreSQL e permite analisar os dados das camadas **Silver** e **Gold**.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import sqlalchemy
from sqlalchemy.engine import URL
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# configs visuais
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12

def load_env_file():
    for env_path in (Path.cwd() / '.env', Path.cwd().parent / '.env'):
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, value = line.split('=', 1)
                    os.environ.setdefault(key.strip(), value.strip())
            break

load_env_file()

# conexao com o banco
db_url = URL.create(
    drivername='postgresql',
    username=os.environ.get('POSTGRES_USER', 'postgres'),
    password=os.environ.get('POSTGRES_PASSWORD', 'postgres'),
    host='localhost',
    port=5432,
    database=os.environ.get('NY_TAXI_DB_NAME', 'ny_taxi'),
)
engine = sqlalchemy.create_engine(db_url)
with engine.connect() as conn:
    pass
print('conexao com o banco estabelecida com sucesso')

---
## 1. camada silver

In [ ]:
# total de linhas e separacao por validas e invalidas
df_qualidade = pd.read_sql("""
    SELECT
        CASE WHEN is_valid_trip THEN 'Validas' ELSE 'Invalidas' END AS status,
        COUNT(*) AS total
    FROM silver.trips
    GROUP BY is_valid_trip
""", engine)

df_qualidade['percentual'] = (df_qualidade['total'] * 100.0 / df_qualidade['total'].sum()).round(2)

print('qualidade dos dados (silver.trips)')
print(f"Total de registros: {df_qualidade['total'].sum():,}")
display(df_qualidade)

In [ ]:
# principais motivos de invalidacao
df_motivos = pd.read_sql("""
    SELECT
        invalid_reason,
        COUNT(*) AS total
    FROM silver.trips
    WHERE is_valid_trip = false
    GROUP BY invalid_reason
    ORDER BY total DESC
    LIMIT 10
""", engine)

print('top 10 motivos de invalidacao')
display(df_motivos)

In [ ]:
# tipos de pagamento
df_payment = pd.read_sql("SELECT * FROM silver.payment_types ORDER BY payment_type", engine)
print('Tipos de pagamento — silver.payment_types')
display(df_payment)

---
## 2. camada Gold — indicadores mensais

In [ ]:
# Carregar view de indicadores mensais
df_gold = pd.read_sql("""
    SELECT *
    FROM gold.mv_monthly_indicators
    WHERE month_yyyymm >= '202501'
    ORDER BY month_yyyymm, vendor_id
""", engine)

print('gold.mv_monthly_indicators')
display(df_gold)

---
## 3. camada gold — ana[lise da tabela fato fct_trips

In [ ]:
# amostra da tabela 
df_fct = pd.read_sql("""
    SELECT *
    FROM gold.fct_trips
    WHERE is_valid_trip = true
    LIMIT 10
""", engine)

print('Amostra de gold.fct_trips (viagens validas)')
display(df_fct)

In [ ]:
# distribuicao de distancia das viagens validas
df_dist = pd.read_sql("""
    SELECT 
        ROUND(trip_distance::numeric, 0) AS distancia_milhas,
        COUNT(*) AS total
    FROM gold.fct_trips
    WHERE is_valid_trip = true
      AND trip_distance BETWEEN 0 AND 30
    GROUP BY distancia_milhas
    ORDER BY distancia_milhas
""", engine)

plt.figure(figsize=(14, 5))
plt.bar(df_dist['distancia_milhas'], df_dist['total'], color='steelblue', edgecolor='white')
plt.title('distribuicao de distancia das viagens validas (0-30 milhas)', fontweight='bold')
plt.xlabel('distancia (milhas)')
plt.ylabel('quantidade de Corridas')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# forma de pagamento mais usada
df_pay = pd.read_sql("""
    SELECT 
        pt.payment_description,
        COUNT(t.payment_type) AS total_corridas,
        ROUND(AVG(t.total_amount)::numeric, 2) AS ticket_medio
    FROM gold.fct_trips t
    JOIN gold.dim_payment_types pt ON t.payment_type = pt.payment_type
    GROUP BY pt.payment_description
    ORDER BY total_corridas DESC
""", engine)

print('corridas e ticket medio por forma de pagamento')
display(df_pay)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_pay['payment_description'], df_pay['total_corridas'], color=sns.color_palette('Set2', len(df_pay)))
ax.set_title('total de corridas por forma de pagamento', fontweight='bold')
ax.set_xlabel('total de corridas')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

---
## 4. Query Livre

use a celula abaixo para rodar qualquer query que quiser no banco

In [ ]:
query = """
    SELECT *
    FROM gold.mv_monthly_indicators
    ORDER BY month_yyyymm
"""

df_custom = pd.read_sql(query, engine)
display(df_custom)